# **Third Homework Assignment**

In [1]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

In [3]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 3)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1, minmax="min"):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0 and minmax == "min":
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        if axis >= 0 and minmax == "max":
            styled_df = styled_df.highlight_max(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_min(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)
    
# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)


In [4]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Load results**

In [5]:
import glob

sizes = [1000, 2000, 4000, 8000]
blocks = [32, 64, 128, 256, 512]

_csv = pd.concat([pd.read_csv(f) for f in glob.glob("*.csv")], ignore_index=True)
_csv.columns = [c.strip() for c in _csv.columns]
_csv["Time"] = pd.to_numeric(_csv["Time"], errors="coerce")
_csv = _csv.dropna(subset=["Time"])

_means = _csv.groupby(["N", "Method", "Cores", "Block"])["Time"].mean()

def _t(method, n, cores=1, block=128):
    key = (n, method, cores, block)
    return float(_means[key]) if key in _means.index else np.nan

## **Basic CUDA comparisons**

**Absolute execution time** *(s)* of the base GPU variant, and **speed-up** against the sequential baseline which was optimized using Newton's third law.

In [6]:
# Absolute time: rows = N, columns = block size.
gpu_time = pd.DataFrame(
    {b: [_t("gpu", n, 1, b) for n in sizes] for b in blocks},
    index=sizes,
)
gpu_time.attrs["name"] = "Base CUDA absolute times"

# Sequential baseline per N.
seq_time = {n: _t("seq", n, 1, 128) for n in sizes}

# Speed-up vs sequential.
gpu_spd = pd.DataFrame(
    {b: [seq_time[n] / _t("gpu", n, 1, b) for n in sizes] for b in blocks},
    index=sizes,
)
gpu_spd.attrs["name"] = "Base CUDA speed-up from sequential"

display_list([gpu_time], axis=1, minmax="min")
display_list([gpu_spd], axis=1, minmax="max")

# Best block per N (used in the method-comparison tables below).
best_gpu_block = {n: int(gpu_time.loc[n].idxmin()) for n in sizes}

,32,64,128,256,512
1000,1.888,1.867,1.789,1.81,1.952
2000,3.65,3.436,3.433,3.491,3.793
4000,6.879,7.026,6.645,6.902,7.443
8000,13.662,13.651,13.849,13.686,14.856


,32,64,128,256,512
1000,4.287,4.335,4.525,4.474,4.147
2000,8.289,8.805,8.812,8.666,7.976
4000,17.278,16.917,17.889,17.223,15.971
8000,33.754,33.781,33.3,33.696,31.042


## **Optimization comparisons**

Comparing the **CUDA base implementation** with the best blocksize which was **128** with **sequential optimized**, **OMP** with **32** cores, **CUDA using neighbourhoods** *(cells)* and **other optimizations** like SoA layout and shared-memory tiling...

In [7]:
omp_cores = [2, 4, 8, 16, 32]

# Best OMP cores and best OPT block per N.
best_omp_cores = {}
best_opt_block = {}
for n in sizes:
    omp_times = {c: _t("omp", n, c, 128) for c in omp_cores}
    omp_times = {c: t for c, t in omp_times.items() if not pd.isna(t)}
    best_omp_cores[n] = min(omp_times, key=omp_times.get)

    opt_times = {b: _t("opt", n, 1, b) for b in blocks}
    opt_times = {b: t for b, t in opt_times.items() if not pd.isna(t)}
    best_opt_block[n] = min(opt_times, key=opt_times.get)

print("Best OMP cores per N:", best_omp_cores)
print("Best OPT block per N:", best_opt_block)
print("Best GPU block per N:", best_gpu_block)

rows_time = []
rows_spd = []
for n in sizes:
    t_base = seq_time[n]
    t_omp = _t("omp", n, best_omp_cores[n], 128)
    t_cuda = _t("gpu", n, 1, best_gpu_block[n])
    t_cells = _t("cells", n, 1, 128)
    t_opt = _t("opt", n, 1, best_opt_block[n])
    rows_time.append({
        "BASE": t_base,
        "OMP": t_omp,
        "CUDA": t_cuda,
        "CELLS": t_cells,
        "OPT": t_opt,
    })
    rows_spd.append({
        "BASE": 1.0,
        "OMP": t_base / t_omp,
        "CUDA": t_base / t_cuda,
        "CELLS": t_base / t_cells,
        "OPT": t_base / t_opt,
    })

cmp_time = pd.DataFrame(rows_time, index=sizes)
cmp_time.attrs["name"] = (
    "Method comparison: absolute times"
)

cmp_spd = pd.DataFrame(rows_spd, index=sizes)
cmp_spd.attrs["name"] = (
    "Method comparison: speed-up"
)

display_list([cmp_time], axis=1, minmax="min")
display_list([cmp_spd], axis=1, minmax="max")

Best OMP cores per N: {1000: 32, 2000: 32, 4000: 32, 8000: 32}
Best OPT block per N: {1000: 128, 2000: 64, 4000: 128, 8000: 128}
Best GPU block per N: {1000: 128, 2000: 128, 4000: 128, 8000: 64}


,BASE,OMP,CUDA,CELLS,OPT
1000,8.096,0.907,1.789,0.415,1.39
2000,30.252,1.766,3.433,0.413,2.524
4000,118.865,3.895,6.645,0.438,4.725
8000,461.153,13.212,13.651,0.794,9.287


,BASE,OMP,CUDA,CELLS,OPT
1000,1.0,8.924,4.525,19.526,5.822
2000,1.0,17.13,8.812,73.179,11.988
4000,1.0,30.514,17.889,271.133,25.159
8000,1.0,34.904,33.781,580.797,49.657
